<a href="https://colab.research.google.com/github/HugoDelgado/AI4MLCysec/blob/main/Module2_AI4CYSEC_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Techniques

This is an exploratory activity where we will practice standardization and Principal Component Analysis (PCA) using the dataset for IoT Network Intrussion Detection from the Module 1 Example Notebook.

#Get and prepare the data

In this cell we consolidated all the tasks needed to download the dataset, load it into a pandas data frame and clean the data. Recall that in Jupyter Notebooks the exclamation sign `!` precedes commands that take place in the underlying Linux operative system.

We describe the data to see its mean and variance before standardizing.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
# Download dataset
! rm -rf *.csv*
! wget -q https://raw.githubusercontent.com/notsodubeyous/IoT-Network-Intrusion-Detection-System-UNSW-NB15/refs/heads/master/datasets/UNSW_NB15.csv

# Load dataset
data = pd.read_csv('UNSW_NB15.csv',header=0)
df = pd.DataFrame(data)

# Clean data
numeric_cols = ['dur', 'sbytes', 'spkts']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=numeric_cols)
df.describe()

# Standardizing the data

As you can observe the mean and standard deviation of the data sets is quite varied. Lets proceed now to use `Standard Scaler` from the `sklearn.preprocesssing` library. And then lets look at the standardized dataset.

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Eliminate the ID column which is not a feature
df.drop(columns=['id'], inplace=True)

# Consider only those values that are numeric
X = df.select_dtypes(include=[np.number]).to_numpy()
scaler = StandardScaler()
X_standardized = scaler.fit_transform(X)

# Convert the standardized NumPy array back to a DataFrame for description
X_standardized_df = pd.DataFrame(X_standardized, columns=df.select_dtypes(include=[np.number]).columns)

# Added a style formating so that we can see the results
X_standardized_df.describe().style.format("{:.4f}")

# Principal Component Analysis

To do Principal Component Analysis (PCA) and reduce dimensionality we can use the `PCA` module in the `sklearn.decomposition` library. Lets work on our standardized dataframe which is required for PCA.

In [ ]:
from sklearn.decomposition import PCA

# Instantiate a PCA ad use it to reduce the dimensionality of the data
pca = PCA()
pca.fit_transform(X_standardized_df)
print(pca.explained_variance_ratio_)

# Visualizing the Result

Using `matplotlib` we get a sense of hoy are column correlated. The sum of the full array should be equal to 1. The array `pca.explained_variance_ratio_` shows how much of the variance is accounted for in each direction. We can choose a subset of the columns to get a significant amount of the variance and model the behavior of our data with combinations of these features.

In [ ]:
import matplotlib.pyplot as plt
data = pca.explained_variance_ratio_
x_values = np.arange(len(data))

# Define the categories
categories = X_standardized_df.columns.to_numpy()

# Create the bar plot
plt.figure(figsize=(10,5))
plt.bar(x_values, data, color='skyblue', edgecolor='black')

# Replace x-axis numbers with your category names
#plt.xticks(x_values, categories)

# 6. Add titles and labels
plt.title('PCA Analysis')
plt.xlabel('Column')
plt.ylabel('Variance Ratio')

# 7. Display the chart
plt.show()

# How many features is enough?

To examine the result, we can do a cumulative sum of the variances and decide how many features we need. Notice in our data that the first 15 features, out of 39 numeric features, hold 90% of the variance.Therefore many of the other features may not be independent and are probably closely correlated. We could gain performance by using the first 15 numeric features without loosing a representative model of the data.

In [ ]:
total = 0
cumulativesum = [total := total + x for x in data]
cumulative_readable = [f"{x:.4f}" for x in cumulativesum]
for i in range(len(cumulative_readable)):
  print(i,cumulative_readable[i])